# Module 1 · Chapter 3 — 박스플롯과 Tukey의 그림

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 5수치 요약, Tukey fence, 박스플롯 vs 히스토그램 vs 바이올린플롯
- **이 노트북**: 스트리밍 플랫폼 장르별 영화 평점으로 시각화 비교

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 시나리오 데이터

스트리밍 서비스 3개 장르의 합성 영화 평점 데이터를 만듭니다.
- **액션**: 중간 평점, 분산 보통, 저평점 이상치 존재
- **로맨스**: 높은 평점, 분산 작음 (팬층 두꺼움)
- **다큐멘터리**: 평점이 양극단에 몰림 (호불호 강함, 양봉형)

In [ ]:
n = 200

# 액션: 정규 분포, 저평점 이상치 5개
action = rng.normal(3.5, 0.7, n)
action[:5] = rng.uniform(0.5, 1.2, 5)  # 저평점 이상치
action = np.clip(action, 0.5, 5.0)

# 로맨스: 높은 평점, 좁은 분포
romance = rng.normal(4.1, 0.4, n)
romance = np.clip(romance, 1.0, 5.0)

# 다큐멘터리: 양봉 분포 (1.5점대 + 4.5점대)
docu = np.concatenate([
    rng.normal(1.8, 0.4, n // 2),
    rng.normal(4.5, 0.3, n // 2)
])
docu = np.clip(docu, 0.5, 5.0)

df = pd.DataFrame({
    "평점": np.concatenate([action, romance, docu]),
    "장르": ["액션"] * n + ["로맨스"] * n + ["다큐멘터리"] * n,
})

df.groupby("장르")["평점"].agg(["mean", "median", "std"]).round(3)

## 2. 직관 — 5수치 요약 직접 계산

박스플롯을 그리기 전에, 숫자를 손으로 계산해서 그림과 연결합니다.

In [ ]:
def five_number_summary(data, label):
    q1 = np.percentile(data, 25)
    median = np.median(data)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    # whisker 끝: fence 안쪽에서 가장 극단적인 실제 값
    lower_whisker = data[data >= lower_fence].min()
    upper_whisker = data[data <= upper_fence].max()
    outliers = data[(data < lower_fence) | (data > upper_fence)]

    print(f"\n[{label}]")
    print(f"  최솟값(whisker 끝): {lower_whisker:.2f}")
    print(f"  Q1                : {q1:.2f}")
    print(f"  중앙값            : {median:.2f}")
    print(f"  Q3                : {q3:.2f}")
    print(f"  최댓값(whisker 끝): {upper_whisker:.2f}")
    print(f"  IQR               : {iqr:.2f}")
    print(f"  Tukey fence       : [{lower_fence:.2f}, {upper_fence:.2f}]")
    print(f"  이상치 수         : {len(outliers)}개")

for genre, grp in df.groupby("장르"):
    five_number_summary(grp["평점"].values, genre)

## 3. 손으로 계산해 보기 — Tukey fence와 이상치 추출

In [ ]:
# 액션 장르의 이상치를 직접 필터링
action_series = df[df["장르"] == "액션"]["평점"]
q1, q3 = action_series.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = action_series[(action_series < lower_fence) | (action_series > upper_fence)]
print(f"액션 장르 이상치 ({len(outliers)}개):")
print(outliers.values.round(2))

## 4. 라이브러리로 같은 답 재현 — 세 가지 시각화 비교

같은 데이터를 박스플롯·히스토그램·바이올린플롯으로 그립니다.
각 시각화가 *무엇을 보여 주고 무엇을 숨기는지* 를 관찰합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = {"액션": "steelblue", "로맨스": "indianred", "다큐멘터리": "mediumseagreen"}

# 박스플롯
sns.boxplot(data=df, x="장르", y="평점", palette=palette, ax=axes[0])
axes[0].set_title("박스플롯\n(이상치 보임, 양봉 숨김)", fontsize=11)

# 히스토그램
for genre, color in palette.items():
    grp = df[df["장르"] == genre]["평점"]
    axes[1].hist(grp, bins=25, alpha=0.5, label=genre, color=color)
axes[1].set_title("히스토그램\n(분포 모양 보임)", fontsize=11)
axes[1].set_xlabel("평점")
axes[1].legend()

# 바이올린플롯
sns.violinplot(data=df, x="장르", y="평점", palette=palette,
               inner="box", ax=axes[2])
axes[2].set_title("바이올린플롯\n(분포 모양 + 박스 요약)", fontsize=11)

plt.suptitle("같은 데이터, 세 가지 시각화", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 함정 점검 — "같은 박스플롯, 다른 분포"

IQR과 중앙값이 같은 네 가지 분포를 만들어 봅니다.
박스플롯은 같지만, 실제 분포는 전혀 다릅니다.

In [ ]:
rng0 = np.random.default_rng(seed=0)
n_pts = 300

# 목표: median≈3.0, Q1≈2.5, Q3≈3.5 (IQR≈1.0)
dist_normal   = rng0.normal(3.0, 0.5, n_pts)
dist_uniform  = rng0.uniform(2.5, 3.5, n_pts)
dist_bimodal  = np.concatenate([rng0.normal(2.6, 0.2, n_pts//2),
                                  rng0.normal(3.4, 0.2, n_pts//2)])
dist_skewed   = np.concatenate([rng0.normal(2.9, 0.3, int(n_pts*0.9)),
                                  rng0.uniform(3.5, 4.5, int(n_pts*0.1))])

labels = ["정규", "균등", "양봉", "우편포"]
dists  = [dist_normal, dist_uniform, dist_bimodal, dist_skewed]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, (d, lbl) in enumerate(zip(dists, labels)):
    axes[0, i].boxplot(d, vert=True)
    axes[0, i].set_title(lbl, fontsize=11)
    axes[0, i].set_ylim(1.5, 5.0)
    axes[0, i].set_xticks([])

    axes[1, i].hist(d, bins=30, color="steelblue", alpha=0.7)
    axes[1, i].set_xlabel("값")

axes[0, 0].set_ylabel("박스플롯")
axes[1, 0].set_ylabel("히스토그램")
plt.suptitle("박스플롯은 비슷하지만 실제 분포는 전혀 다름", fontsize=13)
plt.tight_layout()
plt.show()

print("박스플롯만 보고 '분포가 같다'고 결론 내리지 마세요.")
print("특히 다큐멘터리처럼 양봉 분포는 박스플롯에서 드러나지 않습니다.")

## 6. 바이올린플롯 — 그룹 크기(n)를 폭으로 표현하기

박스플롯은 그룹 크기를 보여 주지 않습니다. 그룹 크기가 다를 때
바이올린플롯의 폭(`density_norm='count'`)을 사용하면
데이터 수가 많은 그룹이 더 넓게 표현되어 비교가 직관적입니다.

> 예: 액션 200편 vs 로맨스 200편 vs 다큐멘터리 **50편** 으로 크기를 다르게 만들어 비교합니다.

In [ ]:
# 그룹 크기를 의도적으로 다르게 생성
rng3 = np.random.default_rng(seed=99)
n_action, n_romance, n_docu_small = 200, 200, 50

action_s  = np.clip(rng3.normal(3.5, 0.7, n_action), 0.5, 5.0)
romance_s = np.clip(rng3.normal(4.1, 0.4, n_romance), 1.0, 5.0)
docu_s    = np.clip(
    np.concatenate([rng3.normal(1.8, 0.4, n_docu_small // 2),
                    rng3.normal(4.5, 0.3, n_docu_small // 2)]),
    0.5, 5.0
)

df_unequal = pd.DataFrame({
    "평점": np.concatenate([action_s, romance_s, docu_s]),
    "장르": (["액션"] * n_action + ["로맨스"] * n_romance
              + ["다큐멘터리"] * n_docu_small),
})

print("그룹별 샘플 크기:")
print(df_unequal["장르"].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette2 = {"액션": "steelblue", "로맨스": "indianred", "다큐멘터리": "mediumseagreen"}

# 기본 바이올린 (폭 고정)
sns.violinplot(data=df_unequal, x="장르", y="평점",
               palette=palette2, inner="box", ax=axes[0])
axes[0].set_title("바이올린플롯 (폭 고정)\n그룹 크기가 다른지 알 수 없음", fontsize=11)
for ax, (genre, cnt) in zip([axes[0]] * 3,
                              df_unequal["장르"].value_counts().items()):
    pass  # 레이블은 아래에서 일괄 추가

# n 비례 바이올린 (density_norm='count')
sns.violinplot(data=df_unequal, x="장르", y="평점",
               palette=palette2, inner="box",
               density_norm="count", ax=axes[1])
axes[1].set_title("바이올린플롯 (폭 ∝ n)\n넓을수록 데이터가 많음", fontsize=11)

# 각 그룹 n 표시
counts = df_unequal.groupby("장르").size()
order  = ["액션", "로맨스", "다큐멘터리"]
for ax in axes:
    for xi, genre in enumerate(order):
        ax.text(xi, 5.2, f"n={counts[genre]}", ha="center",
                fontsize=10, color="dimgray")
    ax.set_ylim(0.0, 5.6)

plt.suptitle("그룹 크기가 다를 때 — density_norm='count' 를 쓰면 n 이 눈에 보인다", fontsize=12)
plt.tight_layout()
plt.show()

print()
print("→ 왼쪽: 다큐멘터리(50편)와 액션(200편)의 폭이 같아 보여 데이터 크기를 놓칩니다.")
print("→ 오른쪽: density_norm='count' 로 폭을 n 에 비례시키면")
print("          다큐멘터리가 훨씬 얇게 그려져 표본이 적음을 즉시 알 수 있습니다.")

## 7. 직접 답해 보기

1. 박스플롯의 whisker 끝이 최솟값/최댓값이 아니라는 것을 어떻게 설명하겠는가?
2. 다큐멘터리 장르 데이터를 박스플롯만으로 분석했다면 어떤 잘못된 결론을 내릴 수 있었을까?
3. 이 세 가지 시각화(박스플롯·히스토그램·바이올린플롯) 중 내가 가장 자주 쓸 것 같은 것은? 이유는?